In [304]:
import pandas as pd
import numpy as np

In [305]:
df = pd.read_csv("final_df.csv")

In [306]:
df.head(2)

,innings,meta.data_version,meta.created,meta.revision,info.city,info.dates,info.gender,info.match_type,info.match_type_number,info.outcome.winner,...,match_id,info.neutral_venue,info.outcome.by.runs,info.outcome.result,info.outcome.eliminator,info.outcome.method,info.bowl_out,info.outcome.bowl_out,info.supersubs.New Zealand,info.supersubs.South Africa
0,"[{'1st innings': {'team': 'Ireland', 'deliveri...",0.9,2019-09-08,1,Dundee,['2019-09-05'],female,T20,747.0,Bangladesh,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"[{'1st innings': {'team': 'Sri Lanka', 'delive...",0.9,2011-12-08,2,Abu Dhabi,"[datetime.date(2011, 11, 25)]",male,T20,NaN,Pakistan,...,2,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [307]:
df.drop(columns=[
    'meta.data_version',
    'meta.created',
    'meta.revision',
    'info.outcome.bowl_out',
    'info.bowl_out',
    'info.supersubs.South Africa',
    'info.supersubs.New Zealand',
    'info.outcome.eliminator',
    'info.outcome.result',
    'info.outcome.method',
    'info.neutral_venue',
    'info.match_type_number',
    'info.outcome.by.runs',
    'info.outcome.by.wickets'
],inplace=True)

In [308]:
df.shape

(1432, 14)

In [309]:
df["info.gender"].value_counts()

info.gender
male      966
female    466
Name: count, dtype: int64

In [310]:
df = df[df["info.gender"] == "male"]
df.drop(columns=["info.gender"],inplace=True)

In [311]:
df.head(2)

,innings,info.city,info.dates,info.match_type,info.outcome.winner,info.overs,info.player_of_match,info.teams,info.toss.decision,info.toss.winner,info.umpires,info.venue,match_id
1,"[{'1st innings': {'team': 'Sri Lanka', 'delive...",Abu Dhabi,"[datetime.date(2011, 11, 25)]",T20,Pakistan,20,['Aizaz Cheema'],"['Pakistan', 'Sri Lanka']",bat,Sri Lanka,"['Ahsan Raza', 'Zameer Haider']",Sheikh Zayed Stadium,2
2,"[{'1st innings': {'team': 'Ireland', 'deliveri...",Dublin,"[datetime.date(2015, 7, 25)]",T20,Netherlands,20,['Mudassar Bukhari'],"['Ireland', 'Netherlands']",field,Netherlands,"['JD Cloete', 'VA Kulkarni']","The Village, Malahide",3


In [312]:
df["info.match_type"].value_counts()

info.match_type
T20    966
Name: count, dtype: int64

In [313]:
df["info.overs"].value_counts()

info.overs
20    963
50      3
Name: count, dtype: int64

In [314]:
df = df[df["info.overs"] == 20]
df.drop(columns=["info.overs"],inplace=True)

In [315]:
import ast

df['info.teams'] = df['info.teams'].apply(ast.literal_eval)

In [316]:

df['innings'] = df['innings'].apply(ast.literal_eval)

In [317]:
df.iloc[0]['innings'][0]['1st innings']['deliveries']

[{0.1: {'batsman': 'WU Tharanga',
   'bowler': 'Umar Gul',
   'extras': {'wides': 1},
   'non_striker': 'TM Dilshan',
   'runs': {'batsman': 0, 'extras': 1, 'total': 1}}},
 {0.2: {'batsman': 'WU Tharanga',
   'bowler': 'Umar Gul',
   'extras': {'wides': 1},
   'non_striker': 'TM Dilshan',
   'runs': {'batsman': 0, 'extras': 1, 'total': 1}}},
 {0.3: {'batsman': 'WU Tharanga',
   'bowler': 'Umar Gul',
   'non_striker': 'TM Dilshan',
   'runs': {'batsman': 2, 'extras': 0, 'total': 2}}},
 {0.4: {'batsman': 'WU Tharanga',
   'bowler': 'Umar Gul',
   'non_striker': 'TM Dilshan',
   'runs': {'batsman': 0, 'extras': 0, 'total': 0}}},
 {0.5: {'batsman': 'WU Tharanga',
   'bowler': 'Umar Gul',
   'non_striker': 'TM Dilshan',
   'runs': {'batsman': 0, 'extras': 0, 'total': 0}}},
 {0.6: {'batsman': 'WU Tharanga',
   'bowler': 'Umar Gul',
   'non_striker': 'TM Dilshan',
   'runs': {'batsman': 0, 'extras': 0, 'total': 0}}},
 {0.7: {'batsman': 'WU Tharanga',
   'bowler': 'Umar Gul',
   'non_striker':

In [318]:
count = 1
delivery_df = pd.DataFrame()
for index, row in df.iterrows():
    if count in [75,108,150,180,268,360,443,458,584,748,982,1052,1111,1226,1345]:
        count+=1
        continue
    count+=1
    ball_of_match = []
    batsman = []
    bowler = []
    runs = []
    player_of_dismissed = []
    teams = []
    batting_team = []
    match_id = []
    city = []
    venue = []
    for ball in row['innings'][0]['1st innings']['deliveries']:
        for key in ball.keys():
            match_id.append(count)
            batting_team.append(row['innings'][0]['1st innings']['team'])
            teams.append(row['info.teams'])
            ball_of_match.append(key)
            batsman.append(ball[key]['batsman'])
            bowler.append(ball[key]['bowler'])
            runs.append(ball[key]['runs']['total'])
            city.append(row['info.city'])
            venue.append(row['info.venue'])
            try:
                player_of_dismissed.append(ball[key]['wicket']['player_out'])
            except:
                player_of_dismissed.append('0')
    loop_df = pd.DataFrame({
            'match_id':match_id,
            'teams':teams,
            'batting_team':batting_team,
            'ball':ball_of_match,
            'batsman':batsman,
            'bowler':bowler,
            'runs':runs,
            'player_dismissed':player_of_dismissed,
            'city':city,
            'venue':venue
        })
    delivery_df = pd.concat([delivery_df, loop_df], ignore_index=True)

In [319]:
delivery_df

,match_id,teams,batting_team,ball,batsman,bowler,runs,player_dismissed,city,venue
0,2,"[Pakistan, Sri Lanka]",Sri Lanka,0.1,WU Tharanga,Umar Gul,1,0,Abu Dhabi,Sheikh Zayed Stadium
1,2,"[Pakistan, Sri Lanka]",Sri Lanka,0.2,WU Tharanga,Umar Gul,1,0,Abu Dhabi,Sheikh Zayed Stadium
2,2,"[Pakistan, Sri Lanka]",Sri Lanka,0.3,WU Tharanga,Umar Gul,2,0,Abu Dhabi,Sheikh Zayed Stadium
3,2,"[Pakistan, Sri Lanka]",Sri Lanka,0.4,WU Tharanga,Umar Gul,0,0,Abu Dhabi,Sheikh Zayed Stadium
4,2,"[Pakistan, Sri Lanka]",Sri Lanka,0.5,WU Tharanga,Umar Gul,0,0,Abu Dhabi,Sheikh Zayed Stadium
...,...,...,...,...,...,...,...,...,...,...
115288,964,"[New Zealand, Pakistan]",New Zealand,19.3,DL Vettori,Saeed Ajmal,1,0,Barbados,"Kensington Oval, Bridgetown"
115289,964,"[New Zealand, Pakistan]",New Zealand,19.4,NL McCullum,Saeed Ajmal,0,0,Barbados,"Kensington Oval, Bridgetown"
115290,964,"[New Zealand, Pakistan]",New Zealand,19.5,NL McCullum,Saeed Ajmal,0,0,Barbados,"Kensington Oval, Bridgetown"
115291,964,"[New Zealand, Pakistan]",New Zealand,19.6,NL McCullum,Saeed Ajmal,1,DL Vettori,Barbados,"Kensington Oval, Bridgetown"


In [320]:
delivery_df.shape

(115293, 10)

In [321]:
def bowl(row):
    for team in row["teams"]:
        if team != row["batting_team"]:
            return team

In [322]:
delivery_df["bowling_team"] = delivery_df.apply(bowl,axis=1)

In [323]:
delivery_df.drop(columns=["teams"],inplace=True)

In [324]:
delivery_df.shape

(115293, 10)

In [325]:
delivery_df["batting_team"].unique()

<ArrowStringArray>
[               'Sri Lanka',                  'Ireland',
             'South Africa',              'Afghanistan',
                    'Kenya',                 'Pakistan',
                    'Qatar',                  'Bermuda',
                'Australia',                 'Scotland',
                    'India',              'West Indies',
                     'Oman',              'Philippines',
                    'Nepal',              'Netherlands',
         'Papua New Guinea',              'New Zealand',
                  'England',               'Bangladesh',
                 'Thailand',                'Singapore',
                 'Malaysia',                'Hong Kong',
                 'Maldives',                  'Belgium',
                 'Zimbabwe',     'United Arab Emirates',
           'Cayman Islands',                   'Canada',
                  'Denmark',                 'Guernsey',
                  'Nigeria',                   'Jersey',
            

In [326]:
selected_teams = [
    'Australia',
    'India',
    'Bangladesh',
    'New Zealand',
    'South Africa',
    'England',
    'West Indies',
    'Afghanistan',
    'Pakistan',
    'Sri Lanka'
]

In [327]:
print(delivery_df["batting_team"].unique())
print(delivery_df["bowling_team"].unique())

<ArrowStringArray>
[               'Sri Lanka',                  'Ireland',
             'South Africa',              'Afghanistan',
                    'Kenya',                 'Pakistan',
                    'Qatar',                  'Bermuda',
                'Australia',                 'Scotland',
                    'India',              'West Indies',
                     'Oman',              'Philippines',
                    'Nepal',              'Netherlands',
         'Papua New Guinea',              'New Zealand',
                  'England',               'Bangladesh',
                 'Thailand',                'Singapore',
                 'Malaysia',                'Hong Kong',
                 'Maldives',                  'Belgium',
                 'Zimbabwe',     'United Arab Emirates',
           'Cayman Islands',                   'Canada',
                  'Denmark',                 'Guernsey',
                  'Nigeria',                   'Jersey',
            

In [328]:
delivery_df = delivery_df[
    delivery_df["batting_team"].isin(selected_teams)
]

delivery_df = delivery_df[
    delivery_df["bowling_team"].isin(selected_teams)
]

In [329]:
delivery_df.shape

(63612, 10)

In [330]:
output = delivery_df[['match_id','batting_team','bowling_team','ball','runs','player_dismissed','city','venue']]

In [331]:
output.head()

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,venue
0,2,Sri Lanka,Pakistan,0.1,1,0,Abu Dhabi,Sheikh Zayed Stadium
1,2,Sri Lanka,Pakistan,0.2,1,0,Abu Dhabi,Sheikh Zayed Stadium
2,2,Sri Lanka,Pakistan,0.3,2,0,Abu Dhabi,Sheikh Zayed Stadium
3,2,Sri Lanka,Pakistan,0.4,0,0,Abu Dhabi,Sheikh Zayed Stadium
4,2,Sri Lanka,Pakistan,0.5,0,0,Abu Dhabi,Sheikh Zayed Stadium


In [332]:
output.shape

(63612, 8)

In [336]:
output.bowling_team.value_counts()

bowling_team
Australia       8477
Pakistan        7878
England         7773
India           7237
New Zealand     7048
West Indies     6951
Sri Lanka       6853
South Africa    5666
Bangladesh      4772
Afghanistan      957
Name: count, dtype: int64

In [333]:
import pickle

In [334]:
pickle.dump(output,open("dataset_level1.pkl","wb"))